# 1. Check Overlap

In [ ]:
import pandas as pd
from rdkit.Chem import MolFromSmiles, MolToInchiKey
import json

# Load your target datasets
lipo = pd.read_csv('./Lipophilicity_smarts.csv')   # has 'smiles' column
# esol = pd.read_csv('./ESOL.csv')

task_inchikeys = set()
# for smi in pd.concat([lipo['smiles'], esol['smiles']]):  
for smi in lipo['smiles']:
    mol = MolFromSmiles(smi)
    if mol:
        task_inchikeys.add(MolToInchiKey(mol))

# Check your 100k
overlap = 0
with open('./SMARTSGPT_warmup.jsonl') as f:
    for line in f:
        smi = json.loads(line)['smiles']
        mol = MolFromSmiles(smi)
        if mol and MolToInchiKey(mol) in task_inchikeys:
            
            overlap += 1

print(f"Overlap: {overlap}")

# 2. Check Token Vocab

In [1]:
import json

# Load the ACTUAL full vocab file
with open('../data/SMARTSGPT_Pubchem_tokenizer.json') as f:                
    vocab_data = json.load(f)

# Handle either format
if 'token2id' in vocab_data:
    TOKEN2ID = vocab_data['token2id']
else:
    TOKEN2ID = vocab_data

KNOWN_VOCAB = set(TOKEN2ID.keys())
print(f"Vocab loaded: {len(KNOWN_VOCAB)} tokens")     # ← should say ~180
assert len(KNOWN_VOCAB) > 150, "Vocab file loaded incompletely!"

Vocab loaded: 153 tokens


In [2]:
import json, re
from collections import Counter

# Exact same regex the tokenizer uses (from your report)
TOK_RE = re.compile(r'(\[[^\]]+\]|Br|Cl|Si|Se|@@|>>|->)|(.)', re.DOTALL)

def tokenize(s):
    return [m.group() for m in TOK_RE.finditer(s)]

oov_tok_counts   = Counter()   # how often each OOV token type appears
oov_frag_counts  = Counter()   # how many fragments each OOV token type corrupts
total_tokens     = 0
total_frags      = 0
corrupt_frags    = 0

with open('sft_warmup.jsonl') as f:
    for line in f:
        item = json.loads(line)
        for frag in item['fragments']:
            total_frags += 1
            tokens = tokenize(frag)
            total_tokens += len(tokens)

            oov_here = set()
            for tok in tokens:
                if tok not in KNOWN_VOCAB:
                    oov_tok_counts[tok] += 1
                    oov_here.add(tok)

            if oov_here:
                corrupt_frags += 1
                for tok in oov_here:
                    oov_frag_counts[tok] += 1

# ── Summary ────────────────────────────────────────────────────────────────
print(f"Vocabulary size:       {len(KNOWN_VOCAB)}")
print(f"Total fragments:       {total_frags:,}")
print(f"Corrupt fragments:     {corrupt_frags:,}  ({100*corrupt_frags/total_frags:.1f}%)")
print(f"Total token positions: {total_tokens:,}")
print(f"OOV token occurrences: {sum(oov_tok_counts.values()):,}")
print(f"Distinct OOV types:    {len(oov_tok_counts)}")

print("\n── Top OOV tokens (by occurrence) ──")
print(f"{'Token':<35} {'occurrences':>12}  {'frags affected':>14}")
for tok, n in oov_tok_counts.most_common(40):
    print(f"  {tok:<33} {n:>12,}  {oov_frag_counts[tok]:>14,}")

Vocabulary size:       153
Total fragments:       1,976,147
Corrupt fragments:     0  (0.0%)
Total token positions: 7,127,254
OOV token occurrences: 0
Distinct OOV types:    0

── Top OOV tokens (by occurrence) ──
Token                                occurrences  frags affected
